In [ ]:
# Input: loco_species_input_427_original_order.xlsx, sheet Merged, with 427 sample rows and 503 numeric species columns.
# Input columns: Metaphlan_ID, optional Disease/Sample_ID/Full_name, and TA, IAM, ILA, I3C, IE, IAA, IA, IPA, INDOLE.
# Metaphlan_ID must be unique; its prefix identifies the cohort and must match the configured cohort counts.

from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, t
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

arial_candidates = [
    Path(r"C:\Windows\Fonts\arial.ttf"),
    Path(r"C:\Windows\Fonts\Arial.ttf"),
]
arial_file = next((path for path in arial_candidates if path.exists()), None)
if arial_file is not None:
    font_manager.fontManager.addfont(str(arial_file))

try:
    resolved_arial = font_manager.findfont("Arial", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError(
        "Arial was not found. Install Arial or update arial_candidates."
    ) from exc

mpl.rcParams.update({
    "font.family": "Arial",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.unicode_minus": False,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})

INPUT_NAME = "loco_species_input_427_original_order.xlsx"
SHEET_NAME = "Merged"
OUTPUT_DIR_NAME = "loco_scatter_grid"
OUTPUT_NAME = "loco_scatter_grid.pdf"
SPEARMAN_TABLE_NAME = "loco_spearman_summary.xlsx"
SHOW_FIGURE = True

TARGETS = ["TA", "IAM", "ILA", "I3C", "IE", "IAA", "IA", "IPA", "INDOLE"]
COHORTS = ["CRC", "cxy", "T2D", "VD_Aging", "va", "Yangzixin"]
COHORT_RENAME = {"VD": "VD_Aging", "Aging": "VD_Aging"}
COHORT_STYLES = {
    "CRC": {"label": "Study 1", "color": "#E64B35"},
    "cxy": {"label": "Study 2", "color": "#9F79B6"},
    "T2D": {"label": "Study 3", "color": "#3C5488"},
    "VD_Aging": {"label": "Study 4", "color": "#00A087"},
    "va": {"label": "Study 5", "color": "#D07D07"},
    "Yangzixin": {"label": "Study 6", "color": "#68AADC"},
}
TARGET_DISPLAY_LABELS = {"INDOLE": "Indole"}
EXPECTED_COHORT_COUNTS = {
    "CRC": 21,
    "cxy": 114,
    "T2D": 17,
    "VD_Aging": 39,
    "va": 13,
    "Yangzixin": 223,
}
STABLE_COHORT_SEED_INDEX = {
    "CRC": 1,
    "cxy": 2,
    "T2D": 3,
    "VD_Aging": 4,
    "va": 6,
    "Yangzixin": 7,
}
MODEL_RANDOM_STATE = 20260806
LOG_PSEUDOCOUNT = 1.0
MIN_TEST_N = 3

def locate_local_input(filename):
    path = Path(filename).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Input file not found: {path}. Place it alongside the notebook or set INPUT_NAME to its path.")
    return path

INPUT_PATH = locate_local_input(INPUT_NAME)
BASE_DIR = INPUT_PATH.parent
OUTPUT_DIR = BASE_DIR / OUTPUT_DIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_PATH.name}")
print(f"Output directory: {OUTPUT_DIR.name}")
print(f"Arial: {Path(resolved_arial).name}")
print(f"Matplotlib PDF font type: {mpl.rcParams['pdf.fonttype']}")


In [ ]:
LOCKED_MODELS = {
    "TA": "XGBoost",
    "IAM": "RandomForest",
    "ILA": "XGBoost",
    "I3C": "XGBoost",
    "IE": "RandomForest",
    "IAA": "RandomForest",
    "IA": "RandomForest",
    "IPA": "ExtraTrees",
    "INDOLE": "RandomForest",
}

def make_model_templates():
    return {
        "RandomForest": RandomForestRegressor(
            n_estimators=250,
            max_features=0.5,
            min_samples_leaf=2,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=250,
            max_features=1.0,
            min_samples_leaf=2,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        "XGBoost": XGBRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
    }

model_templates = make_model_templates()
data = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)
metadata_columns = ["Metaphlan_ID", "Disease", "Sample_ID", "Full_name"]
required_columns = {"Metaphlan_ID", *TARGETS}
missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns[:10]}")
if data["Metaphlan_ID"].isna().any() or data["Metaphlan_ID"].duplicated().any():
    raise ValueError("Metaphlan_ID values must be non-missing and unique.")
if len(data) != 427:
    raise ValueError(f"Expected 427 samples, found {len(data)}.")

feature_columns = [
    column
    for column in data.columns
    if column not in set(metadata_columns + TARGETS)
]
if len(feature_columns) != 503:
    raise ValueError(f"Expected 503 species features, found {len(feature_columns)}.")

cohort_series = (
    data["Metaphlan_ID"]
    .astype(str)
    .str.split("_", n=1)
    .str[0]
    .replace(COHORT_RENAME)
)
actual_cohort_counts = cohort_series.value_counts().to_dict()
if actual_cohort_counts != EXPECTED_COHORT_COUNTS:
    raise ValueError(
        f"Unexpected cohort counts: {actual_cohort_counts}; "
        f"expected {EXPECTED_COHORT_COUNTS}."
    )

x_raw = data[feature_columns].apply(pd.to_numeric, errors="coerce")
negative_x = int((x_raw < 0).sum().sum())
if negative_x:
    raise ValueError(f"Species features contain {negative_x} negative values.")
x_log2 = np.log2(x_raw.fillna(0.0).to_numpy(dtype=float) + LOG_PSEUDOCOUNT)
cohort_array = cohort_series.to_numpy(dtype=object)

print(f"Samples: {len(data)}")
print(f"Species features: {len(feature_columns)}")
print("Models: " + ", ".join(f"{target}={LOCKED_MODELS[target]}" for target in TARGETS))


In [ ]:
def safe_spearman(y_true, y_pred):
    if (
        len(y_true) < 2
        or np.unique(y_true).size < 2
        or np.unique(y_pred).size < 2
    ):
        return np.nan, np.nan
    result = spearmanr(y_true, y_pred)
    return float(result.statistic), float(result.pvalue)

def seeded_clone(estimator, seed):
    model = clone(estimator)
    if "random_state" in model.get_params(deep=False):
        model.set_params(random_state=seed)
    return model

prediction_rows = []
skipped_rows = []

for target_number, target in enumerate(TARGETS, start=1):
    y_raw = pd.to_numeric(data[target], errors="coerce").to_numpy(dtype=float)
    valid_mask = np.isfinite(y_raw)
    if np.any(y_raw[valid_mask] < 0):
        raise ValueError(f"{target} contains negative target values.")

    y_transformed = np.full(len(data), np.nan, dtype=float)
    if target == "INDOLE":
        y_transformed[valid_mask] = np.log(y_raw[valid_mask] + LOG_PSEUDOCOUNT)
    else:
        y_transformed[valid_mask] = np.log2(y_raw[valid_mask] + LOG_PSEUDOCOUNT)

    for held_out_cohort in COHORTS:
        train_index = np.flatnonzero(valid_mask & (cohort_array != held_out_cohort))
        test_index = np.flatnonzero(valid_mask & (cohort_array == held_out_cohort))

        if len(test_index) < MIN_TEST_N:
            skipped_rows.append({
                "Target": target,
                "Held_Out_Cohort": held_out_cohort,
                "Test_N": len(test_index),
            })
            continue

        seed = (
            MODEL_RANDOM_STATE
            + target_number * 10_000
            + STABLE_COHORT_SEED_INDEX[held_out_cohort] * 100
        )
        model_name = LOCKED_MODELS[target]
        model = seeded_clone(model_templates[model_name], seed)
        model.fit(x_log2[train_index], y_transformed[train_index])
        predictions = np.asarray(
            model.predict(x_log2[test_index]), dtype=float
        ).reshape(-1)
        observed = y_transformed[test_index]

        for sample_index, observed_value, predicted_value in zip(
            test_index, observed, predictions
        ):
            prediction_rows.append({
                "Metaphlan_ID": str(data.iloc[sample_index]["Metaphlan_ID"]),
                "Target": target,
                "Model": model_name,
                "Held_Out_Cohort": held_out_cohort,
                "Observed_Transformed": float(observed_value),
                "Predicted_Transformed": float(predicted_value),
            })

loco_predictions = pd.DataFrame(prediction_rows)
skipped_cases = pd.DataFrame(skipped_rows)
if loco_predictions.duplicated(["Target", "Metaphlan_ID"]).any():
    raise RuntimeError("Duplicate target-sample prediction rows were generated.")

statistic_rows = []
for target in TARGETS:
    for held_out_cohort in COHORTS:
        pair = loco_predictions.loc[
            (loco_predictions["Target"] == target)
            & (loco_predictions["Held_Out_Cohort"] == held_out_cohort)
        ]
        if len(pair) < MIN_TEST_N:
            continue
        rho, p_value = safe_spearman(
            pair["Observed_Transformed"].to_numpy(dtype=float),
            pair["Predicted_Transformed"].to_numpy(dtype=float),
        )
        statistic_rows.append({
            "Target": target,
            "Model": LOCKED_MODELS[target],
            "Held_Out_Cohort": held_out_cohort,
            "N": len(pair),
            "Spearman_rho": rho,
            "Spearman_P": p_value,
        })

pair_statistics = pd.DataFrame(statistic_rows)
if pair_statistics.duplicated(["Target", "Held_Out_Cohort"]).any():
    raise RuntimeError("Duplicate target-cohort statistics were generated.")
if len(pair_statistics) != 53:
    raise RuntimeError(
        f"Expected 53 evaluable target-cohort pairs, found {len(pair_statistics)}."
    )
if len(skipped_cases) != 1:
    raise RuntimeError(f"Expected one skipped pair, found {len(skipped_cases)}.")

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(p_values.shape, np.nan, dtype=float)
    finite = np.isfinite(p_values)
    if not finite.any():
        return adjusted
    finite_indices = np.flatnonzero(finite)
    order = np.argsort(p_values[finite])
    sorted_p = p_values[finite][order]
    ranks = np.arange(1, len(sorted_p) + 1, dtype=float)
    sorted_adjusted = np.minimum.accumulate((sorted_p * len(sorted_p) / ranks)[::-1])[::-1]
    sorted_adjusted = np.minimum(sorted_adjusted, 1.0)
    adjusted[finite_indices[order]] = sorted_adjusted
    return adjusted

pair_statistics["Spearman_P"] = benjamini_hochberg(
    pair_statistics["Spearman_P"].to_numpy(dtype=float)
)

print(f"Prediction rows: {len(loco_predictions):,}")
print(f"Evaluable target-cohort pairs: {len(pair_statistics)}")
print(f"Skipped target-cohort pairs: {len(skipped_cases)}")
print("Spearman P values: Benjamini-Hochberg FDR-adjusted across evaluable pairs")

summary_table = pair_statistics.copy()
summary_table["Status"] = "ok"
if not skipped_cases.empty:
    skipped_table = skipped_cases.rename(columns={"Test_N": "N"}).copy()
    skipped_table["Model"] = skipped_table["Target"].map(LOCKED_MODELS)
    skipped_table["Spearman_rho"] = np.nan
    skipped_table["Spearman_P"] = np.nan
    skipped_table["Status"] = "skipped_test_n_lt_3"
    skipped_table = skipped_table[summary_table.columns]
    summary_table = pd.concat([summary_table, skipped_table], ignore_index=True)

summary_table = summary_table[[
    "Target",
    "Model",
    "Held_Out_Cohort",
    "N",
    "Spearman_rho",
    "Spearman_P",
    "Status",
]]
target_order = {target: i for i, target in enumerate(TARGETS)}
cohort_order = {cohort: i for i, cohort in enumerate(COHORTS)}
summary_table["_target_order"] = summary_table["Target"].map(target_order)
summary_table["_cohort_order"] = summary_table["Held_Out_Cohort"].map(cohort_order)
summary_table = (
    summary_table.sort_values(["_target_order", "_cohort_order"])
    .drop(columns=["_target_order", "_cohort_order"])
    .reset_index(drop=True)
)
summary_table.to_excel(
    OUTPUT_DIR / SPEARMAN_TABLE_NAME,
    index=False,
    sheet_name="Spearman",
)
print(f"Saved Spearman summary table: {OUTPUT_DIR / SPEARMAN_TABLE_NAME}")


In [ ]:
from matplotlib.ticker import MaxNLocator, MultipleLocator

def ols_mean_confidence_interval(x, y, x_grid, confidence=0.95):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_grid = np.asarray(x_grid, dtype=float)
    n = x.size
    if n < 3:
        raise ValueError("At least three valid samples are required for OLS.")

    design = np.column_stack([np.ones(n), x])
    beta, *_ = np.linalg.lstsq(design, y, rcond=None)
    fitted = design @ beta
    residual = y - fitted
    dof = n - 2
    sxx = np.sum((x - x.mean()) ** 2)
    if sxx <= 0:
        raise ValueError("The x values have no variation; an OLS line cannot be fitted.")

    residual_variance = np.sum(residual ** 2) / dof
    y_grid = beta[0] + beta[1] * x_grid
    se_mean = np.sqrt(
        residual_variance * (1.0 / n + (x_grid - x.mean()) ** 2 / sxx)
    )
    critical = t.ppf(0.5 + confidence / 2.0, dof)
    return y_grid, y_grid - critical * se_mean, y_grid + critical * se_mean

def format_p_value(p_value):
    if not np.isfinite(p_value):
        return r"p = NA"
    return r"p < 0.001" if p_value < 0.001 else rf"p = {p_value:.3f}"

def padded_limits(values, padding_fraction=0.04, minimum_padding=0.05):

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite values were available for axis limits.")

    lower = float(values.min())
    upper = float(values.max())
    span = upper - lower
    padding = max(span * padding_fraction, minimum_padding)
    return lower - padding, upper + padding

def global_x_axis_limits(predictions):

    values = predictions["Observed_Transformed"].to_numpy(dtype=float)
    lower, upper = padded_limits(values, padding_fraction=0.025, minimum_padding=0.10)

    lower = float(np.floor(lower))
    upper = float(np.ceil(upper))
    if upper <= lower:
        upper = lower + 1.0
    return lower, upper

def target_y_axis_limits(predictions, target):

    values = predictions.loc[
        predictions["Target"] == target,
        "Predicted_Transformed",
    ].to_numpy(dtype=float)
    return padded_limits(values, padding_fraction=0.055, minimum_padding=0.10)

def plot_scatter_matrix(predictions, statistics, output_path, show=True):
    mpl.rcParams.update({
        "font.family": "Arial",
        "font.sans-serif": ["Arial"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.75,
        "xtick.major.width": 0.75,
        "ytick.major.width": 0.75,
    })

    n_rows = len(TARGETS)
    n_columns = len(COHORTS)

    figure_width = 12
    figure_height = 12
    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(figure_width, figure_height),
        squeeze=False,
        sharex=True,
    )

    statistics_lookup = statistics.set_index(
        ["Target", "Held_Out_Cohort"]
    )
    x_limits = global_x_axis_limits(predictions)

    for row_index, target in enumerate(TARGETS):
        y_limits = target_y_axis_limits(predictions, target)
        available_cohorts = [
            cohort
            for cohort in COHORTS
            if np.sum(
                (predictions["Target"] == target)
                & (predictions["Held_Out_Cohort"] == cohort)
            ) >= MIN_TEST_N
        ]
        last_available_cohort = available_cohorts[-1] if available_cohorts else None

        for column_index, cohort in enumerate(COHORTS):
            ax = axes[row_index, column_index]
            cohort_style = COHORT_STYLES[cohort]
            pair = predictions.loc[
                (predictions["Target"] == target)
                & (predictions["Held_Out_Cohort"] == cohort),
                ["Observed_Transformed", "Predicted_Transformed"],
            ].replace([np.inf, -np.inf], np.nan).dropna()

            ax.grid(False)
            ax.set_xlim(x_limits)
            ax.set_ylim(y_limits)
            ax.xaxis.set_major_locator(MultipleLocator(2.0))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=3))

            if len(pair) >= MIN_TEST_N:
                x = pair["Observed_Transformed"].to_numpy(dtype=float)
                y = pair["Predicted_Transformed"].to_numpy(dtype=float)

                x_grid = np.linspace(float(np.min(x)), float(np.max(x)), 300)
                y_fit, y_low, y_high = ols_mean_confidence_interval(
                    x, y, x_grid
                )

                ax.fill_between(
                    x_grid,
                    y_low,
                    y_high,
                    color="#193E58",
                    alpha=0.18,
                    linewidth=0,
                    zorder=1,
                )
                ax.plot(
                    x_grid,
                    y_fit,
                    color="#173F5F",
                    linewidth=1.05,
                    linestyle=(0, (6, 4)),
                    zorder=2,
                )
                ax.scatter(
                    x,
                    y,
                    s=17,
                    color=cohort_style["color"],
                    alpha=0.86,
                    edgecolors="#333333",
                    linewidths=0.25,
                    rasterized=True,
                    zorder=3,
                )

                statistic_row = statistics_lookup.loc[(target, cohort)]
                stats_text = (
                    f"ρ = {statistic_row['Spearman_rho']:.3f}\n"
                    f"{format_p_value(statistic_row['Spearman_P'])}"
                )
                ax.text(
                    0.04,
                    0.96,
                    stats_text,
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=8.5,
                    color="#111111",
                    linespacing=1.05,
                    zorder=4,
                )
            else:
                                                                                                    
                if row_index != n_rows - 1:
                    ax.set_visible(False)
                    continue

            if row_index == 0:
                ax.text(
                    0.5,
                    1.045,
                    f"{cohort_style['label']} (n={EXPECTED_COHORT_COUNTS[cohort]})",
                    transform=ax.transAxes,
                    ha="center",
                    va="bottom",
                    fontsize=12.2,
                    fontweight="bold",
                    color="#111111",
                    clip_on=False,
                )

            if cohort == last_available_cohort:
                target_label = TARGET_DISPLAY_LABELS.get(target, target)
                ax.text(
                    1.075,
                    0.5,
                    target_label,
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    rotation=-90,
                    fontsize=12.0,
                    fontweight="bold",
                    color="#111111",
                    clip_on=False,
                )

            show_y_axis = column_index == 0
            ax.tick_params(
                axis="y",
                which="major",
                left=show_y_axis,
                labelleft=show_y_axis,
                labelsize=8.2,
                width=0.75,
                length=3.0,
                pad=2.0,
                direction="out",
            )

            if row_index == n_rows - 1:
                ax.tick_params(
                    axis="x",
                    which="major",
                    bottom=True,
                    labelbottom=True,
                    labelsize=8.5,
                    width=0.75,
                    length=3.0,
                    pad=2.0,
                    direction="out",
                )
            else:
                ax.tick_params(
                    axis="x",
                    which="major",
                    bottom=False,
                    labelbottom=False,
                )

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["left"].set_linewidth(0.75)
            ax.spines["bottom"].set_linewidth(0.75)
            ax.spines["left"].set_color("#222222")
            ax.spines["bottom"].set_color("#222222")

    fig.supxlabel(
        "Measured value (log2-transformed; INDOLE: ln-transformed)",
        fontsize=14.5,
        x=0.50,
        y=0.020,
    )
    fig.supylabel(
        "LOCO-predicted value (log2-transformed; INDOLE: ln-transformed)",
        fontsize=14.5,
        x=0.015,
        y=0.50,
    )
    fig.subplots_adjust(
        left=0.060,
        right=0.930,
        bottom=0.052,
        top=0.968,
        wspace=0.095,
        hspace=0.055,
    )
    fig.savefig(
        output_path,
        format="pdf",
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
        metadata={
            "Title": "LOCO prediction scatter-plot matrix",
            "Subject": "Measured versus LOCO-predicted log2 values by metabolite and cohort",
            "Creator": "Matplotlib",
        },
    )

    if show:
        plt.show()
    plt.close(fig)
    return output_path


In [ ]:
scatter_grid_path = plot_scatter_matrix(
    loco_predictions,
    pair_statistics,
    output_path=OUTPUT_DIR / OUTPUT_NAME,
    show=SHOW_FIGURE,
)
print(f"Saved: {scatter_grid_path}")
